# export

> Export a trained model to ONNX for production deployment, with a numerical check that ONNX Runtime reproduces the PyTorch output.

A dynamic batch axis is kept so the same file serves single-image and batched inference. The verification step catches op/opset mismatches before the model reaches a production line.

In [ ]:
#| default_exp export

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path

import numpy as np
import torch

## Export to ONNX

The dummy input must live on the same device as the model. We export with `dynamo=False` for a stable, widely-supported graph, then load the file back to run the structural checker.

In [ ]:
#| export
def export_onnx(
    model,                          # trained model returning logits
    output_path: str,               # destination .onnx path
    in_channels: int = 3,           # input channel count (must match training)
    input_size: int = 256,          # square input side for the export trace
    opset: int = 18,                # ONNX opset version
    verify: bool = True,            # compare ONNX Runtime output against PyTorch
    task: str = None,               # 'binary' | 'multiclass' | 'multilabel' -- how a consumer should decode `logits`
    class_names: list = None,       # per-channel names in output-channel order, e.g. ['chip', 'void', 'lifted']
    thresholds: list = None,        # per-channel decode threshold(s), multilabel: one per class_name
    hierarchy_pairs: list = None,   # [(sub_idx, super_idx), ...] subset constraints, e.g. [(1, 0), (2, 0)] == void/lifted subset of chip
) -> str:
    """Export `model` to ONNX with a dynamic batch axis; return the path.

    When `task`/`class_names`/`thresholds`/`hierarchy_pairs` are given, they're written into the
    ONNX file's `metadata_props` (plain string key/value pairs) so a downstream consumer -- e.g.
    `void_detection_example_line`'s inference code -- can decode raw `logits` correctly
    without hardcoding task semantics: multilabel heads need independent per-channel sigmoid, not
    softmax+argmax, and any subset constraint (void/lifted ⊆ chip) needs `apply_hierarchy()`.
    """
    import onnx
    model.eval()
    dev = next(model.parameters()).device
    dummy = torch.randn(1, in_channels, input_size, input_size, device=dev)
    with torch.no_grad():
        pt_out = model(dummy).cpu().numpy()
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        model, dummy, output_path,
        input_names=['image'], output_names=['logits'],
        dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=opset, dynamo=False,
    )
    if task is not None or class_names is not None:
        import json
        onnx_model = onnx.load(output_path)
        props = {'task': task or ''}
        if class_names is not None:
            props['class_names'] = json.dumps(list(class_names))
        if thresholds is not None:
            props['thresholds'] = json.dumps(list(thresholds) if hasattr(thresholds, '__len__') else thresholds)
        if hierarchy_pairs is not None:
            props['hierarchy_pairs'] = json.dumps([list(p) for p in hierarchy_pairs])
        onnx.helper.set_model_props(onnx_model, props)
        onnx.save(onnx_model, output_path)
    onnx.checker.check_model(onnx.load(output_path))
    if verify:
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(output_path)
            ort_out = sess.run(['logits'], {'image': dummy.cpu().numpy()})[0]
            max_diff = float(np.abs(pt_out - ort_out).max())
            assert max_diff < 1e-3, f'ONNX output diverges from PyTorch (max diff {max_diff})'
        except ImportError:
            pass  # onnxruntime not installed; structural check still ran
    return output_path

In [ ]:
# test: export a tiny 2-channel model and confirm a valid file is written (no metadata)
import tempfile
import torch.nn as nn
model = nn.Conv2d(2, 1, 3, padding=1)
with tempfile.TemporaryDirectory() as d:
    out = Path(d) / 'm.onnx'
    export_onnx(model, str(out), in_channels=2, input_size=32, verify=True)
    assert out.exists() and out.stat().st_size > 0

# test: multilabel metadata round-trips through the saved ONNX file
import json
import onnx

class ThreeHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 3, 3, padding=1)
    def forward(self, x):
        return self.conv(x)

model3 = ThreeHead()
with tempfile.TemporaryDirectory() as d:
    out = Path(d) / 'm3.onnx'
    export_onnx(
        model3, str(out), in_channels=1, input_size=32, verify=True,
        task='multilabel', class_names=['chip', 'void', 'lifted'],
        thresholds=[0.5, 0.5, 0.5], hierarchy_pairs=[(1, 0), (2, 0)],
    )
    props = {p.key: p.value for p in onnx.load(str(out)).metadata_props}
    assert props['task'] == 'multilabel'
    assert json.loads(props['class_names']) == ['chip', 'void', 'lifted']
    assert json.loads(props['thresholds']) == [0.5, 0.5, 0.5]
    assert json.loads(props['hierarchy_pairs']) == [[1, 0], [2, 0]]

print('export_onnx ✓  dynamic batch axis, ORT parity check, multilabel task/class_names/thresholds/hierarchy_pairs in metadata_props')

## FP32 ONNX regression smoke — curated encoders via `get_model`

Curated aliases (`convnext_tiny`, `efficientnetv2_s`) and raw `tu-*` names must export
through the same `get_model` → `export_onnx` path production uses: FP32 only, dynamic
batch axis, opset 18, metadata-decodable (`task`/`class_names` in `metadata_props`),
verified against PyTorch with ONNX Runtime. `pretrained=False` keeps the smoke
download-free; DINOv3 (`dinov3_vit_small`/`dinov3_vit_base`) FP32 export at
`in_channels=1`, multilabel `num_classes=3` was validated separately (weights pull via
timm hub — heavier than a smoke test). No INT8, no TensorRT.

In [ ]:
#| test
# test: curated encoders export FP32 via get_model + export_onnx and verify against PyTorch
import onnxruntime as ort

from segmentation_tools_pt.model import get_model

for arch in ['convnext_tiny', 'efficientnetv2_s']:
    m = get_model(arch, in_channels=1, num_classes=3, pretrained=False, architecture='unet')
    m.eval()
    x = torch.randn(1, 1, 256, 256)
    with torch.no_grad():
        ref = m(x)
    with tempfile.TemporaryDirectory() as d:
        p = export_onnx(m, str(Path(d) / f'{arch}.onnx'), in_channels=1, input_size=256,
                        task='multilabel', class_names=['chip', 'void', 'lifted'])
        sess = ort.InferenceSession(p, providers=['CPUExecutionProvider'])
        iname = sess.get_inputs()[0].name
        out = sess.run(None, {iname: x.numpy()})[0]
        assert out.shape == ref.shape == (1, 3, 256, 256)
        assert np.abs(out - ref.numpy()).max() < 1e-3
print('curated encoder FP32 ONNX smoke passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()